# Boosting Models

## Shane Waldron

In [23]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

In [24]:
# Load the dataset
df = pd.read_csv('data/train.csv')

In [25]:
# Encoding the target variable
df["Irrigation_Need"] = df["Irrigation_Need"].map({"High": 2,"Medium": 1, "Low": 0})

# Removing the id column as it is not useful for modeling
df.drop("id", axis=1, inplace=True)

In [26]:
# Separating numerical and categorical columns
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
# Removing the target variable from the list of numerical columns
numerical_cols.remove("Irrigation_Need")

In [27]:
# Identifying X and y and splitting the data into training and testing sets
X = df.drop("Irrigation_Need", axis=1)
y = df["Irrigation_Need"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [28]:
# One-hot encoding for categorical variables
preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='passthrough'
)

In [29]:
# Training the XGBoost model with Optuna for hyperparameter tuning


def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),                                
        "tree_method": "hist",
        "eval_metric": "mlogloss",
        "n_jobs": -1,
        'random_state': 42,
    }
    
    xgb_model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(**params))
    ])
    
    scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    study_name="xgb_local",
    storage="sqlite:///optuna_xgb.db",
    load_if_exists=True,
)

# study.optimize(objective, n_trials=20)




[I 2026-04-24 11:55:25,357] Using an existing study with name 'xgb_local' instead of creating a new one.


In [30]:
best_params = study.best_params
print("Best Hyperparameters:", best_params)

Best Hyperparameters: {'n_estimators': 1186, 'max_depth': 5, 'learning_rate': 0.07392909715570185, 'subsample': 0.8435448211257797, 'colsample_bytree': 0.556708915636543, 'min_child_weight': 6, 'reg_alpha': 4.5224790845332776e-05, 'reg_lambda': 1.9087633509163162}


I had to rerun the notebook but didn't want to spend another 40 minutes allowing the study to run with 20 more trials, the first 20 trials are part of a db within GitHub

In [31]:
# Fitting the XGBoost model with the best hyperparameters
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(**best_params))
])
xgb_model.fit(X_train, y_train)
# Evaluating the XGBoost model
y_pred_xgb = xgb_model.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



### Evaluating XGBoost
The XGBoost model is performing very strongly overall, with 98% accuracy and weighted F1 of 0.98, which shows excellent aggregate classification performance. It predicts classes 0 and 1 especially well, with precision and recall both around 0.98 to 0.99, while high irrigation need is clearly the hardest case: its precision remains high at 0.97, but recall drops to 0.92, meaning the model is missing a noticeable share of true high irrigation need observations. The gap between weighted and macro averages also suggests the overall result is helped by the larger class sizes, so the main area for improvement would be better sensitivity on the minority class.

In [32]:
# Modeling with LightGBM and Optuna for hyperparameter tuning
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),                                
        "random_state": 42,
        "verbosity": -1,
    }
    
    lgbm_model = Pipeline(steps=[
        ('preprocessor', preprocessor.set_output(transform='pandas')),
        ('classifier', LGBMClassifier(**params))
    ])
    
    scores = cross_val_score(lgbm_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_local",
    storage="sqlite:///optuna_lgbm.db",
    load_if_exists=True,
)
study_lgbm.optimize(objective_lgbm, n_trials=20)



[I 2026-04-24 11:55:58,446] Using an existing study with name 'lgbm_local' instead of creating a new one.
[I 2026-04-24 11:57:52,797] Trial 27 finished with value: 0.9844444444444445 and parameters: {'n_estimators': 700, 'max_depth': 6, 'learning_rate': 0.04521618858710297, 'subsample': 0.6263825740566065, 'colsample_bytree': 0.6674560499472237, 'min_child_weight': 7, 'reg_alpha': 0.034048801846325256, 'reg_lambda': 1.0362883935240816e-06}. Best is trial 3 with value: 0.9845668934240364.
[I 2026-04-24 11:58:59,274] Trial 28 finished with value: 0.984562358276644 and parameters: {'n_estimators': 373, 'max_depth': 8, 'learning_rate': 0.089689658837699, 'subsample': 0.7750883217513033, 'colsample_bytree': 0.6760238844598175, 'min_child_weight': 6, 'reg_alpha': 0.6189228272020735, 'reg_lambda': 3.161222209481339e-05}. Best is trial 3 with value: 0.9845668934240364.
[I 2026-04-24 12:00:03,831] Trial 29 finished with value: 0.9845804988662132 and parameters: {'n_estimators': 351, 'max_depth'

In [33]:
best_params_lgbm = study_lgbm.best_params
print("Best Hyperparameters for LightGBM:", best_params_lgbm)

Best Hyperparameters for LightGBM: {'n_estimators': 351, 'max_depth': 9, 'learning_rate': 0.08684459743163021, 'subsample': 0.8838895596055906, 'colsample_bytree': 0.6041416018275265, 'min_child_weight': 6, 'reg_alpha': 0.8415670154835752, 'reg_lambda': 6.372183764796225e-05}


In [34]:
# Modeling with LightGBM and Optuna for hyperparameter tuning with a different set of hyperparameters (and trying stratified cv)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    
    lgbm_model = Pipeline(steps=[
        ('preprocessor', preprocessor.set_output(transform='pandas')),
        ('classifier', LGBMClassifier(**params))
    ])
    
    scores = cross_val_score(lgbm_model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    study_name="lgbm_local_2",
    storage="sqlite:///optuna_lgbm2.db",
    load_if_exists=True,
)
study_lgbm.optimize(objective_lgbm, n_trials=20)



[I 2026-04-24 12:21:33,108] Using an existing study with name 'lgbm_local_2' instead of creating a new one.
[I 2026-04-24 12:25:33,706] Trial 20 finished with value: 0.9842426303854875 and parameters: {'n_estimators': 1181, 'max_depth': 9, 'learning_rate': 0.010377641101255875, 'subsample': 0.7598488639653368, 'colsample_bytree': 0.5835073975866102}. Best is trial 12 with value: 0.9843333333333334.
[I 2026-04-24 12:28:17,858] Trial 21 finished with value: 0.9843650793650793 and parameters: {'n_estimators': 943, 'max_depth': 9, 'learning_rate': 0.032322682873877566, 'subsample': 0.7002573701945382, 'colsample_bytree': 0.551941383617381}. Best is trial 21 with value: 0.9843650793650793.
[I 2026-04-24 12:30:57,571] Trial 22 finished with value: 0.984358276643991 and parameters: {'n_estimators': 922, 'max_depth': 9, 'learning_rate': 0.06331888728540984, 'subsample': 0.7137132171581569, 'colsample_bytree': 0.5464782194676194}. Best is trial 21 with value: 0.9843650793650793.
[I 2026-04-24 1

In [35]:
best_params_lgbm2 = study_lgbm.best_params
print("Best Hyperparameters for LightGBM:", best_params_lgbm2)

Best Hyperparameters for LightGBM: {'n_estimators': 940, 'max_depth': 7, 'learning_rate': 0.0460931948374744, 'subsample': 0.7891671118518555, 'colsample_bytree': 0.5451827069318972}


In [36]:
# Fitting the first LightGBM model with the best hyperparameters
lgbm_model = Pipeline(steps=[
    ('preprocessor', preprocessor.set_output(transform='pandas')),
    ('classifier', LGBMClassifier(**best_params_lgbm))
])
lgbm_model.fit(X_train, y_train)
# Evaluating the LightGBM model
y_pred_lgbm = lgbm_model.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

LightGBM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



In [37]:
# Fitting the second LightGBM model with the best hyperparameters
lgbm_model = Pipeline(steps=[
    ('preprocessor', preprocessor.set_output(transform='pandas')),
    ('classifier', LGBMClassifier(**best_params_lgbm2))
])
lgbm_model.fit(X_train, y_train)
# Evaluating the LightGBM model
y_pred_lgbm = lgbm_model.predict(X_test)
print("LightGBM Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

LightGBM Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    110927
           1       0.98      0.98      0.98     71683
           2       0.97      0.92      0.94      6390

    accuracy                           0.98    189000
   macro avg       0.98      0.96      0.97    189000
weighted avg       0.98      0.98      0.98    189000



### LightGBM analysis
LightGBM performs essentially identically to XGBoost on this classification task, with the same overall accuracy, weighted F1, and class-level precision and recall across low, medium, and high irrigation need. Both models classify low and medium irrigation need extremely well, while high irrigation need is the most difficult category: precision remains strong, but recall is lower, meaning both models miss some truly high-need cases. So in practical terms, neither model has a performance advantage here, and the decision between them would likely come down to factors like training speed, tuning convenience, or how they behave on the test set.

In [38]:
test = pd.read_csv("data/test.csv")

# Creating the submission file
test_ids = test["id"]
test.drop("id", axis=1, inplace=True)
test_predictions = lgbm_model.predict(test)
submission = pd.DataFrame({"id": test_ids, "Irrigation_Need": test_predictions})
# Convert numeric predictions back to original labels
submission["Irrigation_Need"] = submission["Irrigation_Need"].map({0: "Low", 1: "Medium", 2: "High"})
submission.to_csv("submission_lgbm.csv", index=False)
submission.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [39]:
test = pd.read_csv("data/test.csv")

# Creating the submission file
test_ids = test["id"]
test.drop("id", axis=1, inplace=True)
test_predictions = xgb_model.predict(test)
submission = pd.DataFrame({"id": test_ids, "Irrigation_Need": test_predictions})
# Convert numeric predictions back to original labels
submission["Irrigation_Need"] = submission["Irrigation_Need"].map({0: "Low", 1: "Medium", 2: "High"})
submission.to_csv("submission_xgb.csv", index=False)
submission.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
